# Functionality Trojan

AI-generated RTL candidate and revision history.

## 1. Setup

In [1]:
from pathlib import Path
import os, re, json, subprocess
from datetime import datetime, timezone
from dotenv import load_dotenv
from portkey_ai import Portkey

ROOT = Path.cwd().resolve()
while not (ROOT / "recovery/recovered_sim.v").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "recovery/recovered_sim.v").is_file(), "Open this notebook inside the project."

load_dotenv(ROOT / ".env", override=True)
MODEL = "@vertexai/gemini-3.1-pro-preview"
VARIANT = "functionality"
SOURCE = ROOT / "recovery/recovered_sim.v"
OUT = ROOT / "rtl/trojans" / VARIANT
LOG = ROOT / "ai_logs/trojans" / VARIANT
COMMON = ROOT / "rtl/trojans/common"
CELLS = Path(os.getenv("ICE40_CELLS_SIM", "/usr/share/yosys/ice40/cells_sim.v"))

for folder in (OUT, LOG, COMMON):
    folder.mkdir(parents=True, exist_ok=True)

renamed, count = re.subn(
    r"\bmodule\s+recovered\s*\(",
    "module recovered_original(",
    SOURCE.read_text(),
    count=1
)
assert count == 1
CORE = COMMON / "recovered_original.v"
CORE.write_text(renamed)

client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
    max_retries=0
)
print("Variant:", VARIANT)
print("Source:", SOURCE.relative_to(ROOT))

Variant: functionality
Source: recovery/recovered_sim.v


## 2. Specification

In [2]:
SPEC = 'Trigger: receive a complete 32-bit SPI word 0xA6F03C59 while NORM_CS_N is low.\nPayload: corrupt only the following 32-bit MISO transfer, without changing cryptographic state.\nThe trigger transaction itself must not be corrupted. Clear the effect after one complete readout.'
print(SPEC)

Trigger: receive a complete 32-bit SPI word 0xA6F03C59 while NORM_CS_N is low.
Payload: corrupt only the following 32-bit MISO transfer, without changing cryptographic state.
The trigger transaction itself must not be corrupted. Clear the effect after one complete readout.


## 3. Generate candidate

In [4]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = ""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=16384, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_002.sv


## 4. Compile

In [5]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 5. Revision 1

In [7]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous RTL compiles but has transaction timing problems.

1. The final bit of the trigger transaction must never be corrupted.
2. Corruption must begin with the first bit of the following
   complete 32-bit MISO transfer.
3. Corrupt exactly one complete transfer, then return to normal.
4. Handle interrupted transactions and chip-select boundaries.
5. Keep the original cryptographic state unchanged.

Check SPI sampling and output timing carefully. Preserve the exact original module port connections.
"""


# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_004.sv


## 6. Compile

In [8]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 7. Revision 2

In [9]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous functionality RTL still activates corruption on the falling SCK edge immediately after the trigger word.

This does not guarantee the trigger transaction remains normal.

Revise the design so that:

1. The entire trigger transaction is unaffected.
2. The trigger is armed after receiving A6F03C59.
3. The payload activates only for the next SPI transaction.
4. Exactly 32 MISO bits are inverted.
5. Corruption is disabled after that transaction.
6. Interrupted transactions are handled safely.
7. Normal encryption and decryption remain unaffected.

Do not assume that moving corruption to negedge SCK automatically solves the transaction-boundary problem.

Preserve the original core and exact package-pin mapping.
"""


# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_005.sv


## 8. Compile

In [10]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 9. Revision 3

In [11]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous functionality RTL correctly delays payload activation until CS deasserts after the trigger transfer.

However, two issues remain:

1. trigger_matched and payload_completed retain stale values
   across CS transitions when no SCK edge occurs.
2. bit_cnt wraps after 32 bits and can incorrectly treat a
   longer CS-low transaction as multiple complete transfers.

Revise the transaction tracking so that:

- A trigger is accepted only for a complete 32-bit
  transaction containing A6F03C59.
- The trigger transaction's MISO output is never modified.
- Only the next valid complete 32-bit transaction is corrupted.
- Corruption ends when that payload transaction finishes.
- Interrupted or malformed transactions cannot reuse stale
  completion flags.
- CS transitions without SCK are handled correctly.
- State does not wrap into another transaction while CS is low.

Ensure synthesizable, race-free state handling. Avoid multiple procedural drivers, inferred latches, and unsafe clocking.

Preserve the original core and its exact package-pin connections.
"""


# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_006.sv


## 10. Compile

In [12]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 11. Revision 4

In [13]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous functionality RTL correctly detects a complete 32-bit trigger transaction at CS deassertion and avoids corrupting the trigger transaction itself.

Remaining issues:

1. The armed flag only resets on a CS rising edge.
   RST_N must clear the Trojan state reliably even if
   CS does not transition.

2. An interrupted payload transaction may be partially
   corrupted while armed remains set. Define clear behavior
   for interrupted transactions.

Requirements:
- Recognize A6F03C59 only after exactly 32 received bits.
- Never corrupt the trigger transaction.
- Corrupt the following valid 32-bit readout.
- Return to normal after the payload transaction.
- Handle malformed or interrupted transactions predictably.
- Preserve normal cryptographic operation.
- Use synthesizable, FPGA-compatible logic.
- Preserve the original core and all port mappings.

Avoid multiple procedural drivers and unsafe clocking.
"""


# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_007.sv


## 12. Compile

In [14]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 13. Revision 5

In [15]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous functionality implementation reintroduced a trigger-to-payload timing problem.

The armed flag becomes high on the 32nd rising SCK edge and directly controls MISO while CS is still low.

This can change the final bit of the trigger transaction.

Revise the implementation so that:

1. Recognize A6F03C59 only as an exactly 32-bit SPI
   transaction.
2. Do not modify MISO at any point during that transaction.
3. Activate corruption only after the trigger transaction
   has ended and the next transaction begins.
4. Corrupt exactly 32 bits of the next valid readout.
5. Clear the payload after that readout.
6. Handle interrupted or longer-than-32-bit transactions.
7. Ensure reset clears all Trojan state.
8. Avoid multiple procedural drivers and unsafe clocking.

SPI mode is CPOL=0, CPHA=0. The master samples MISO on rising SCK edges.

Preserve the recovered_original module and its pin mapping.
"""


# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_008.sv


## 14. Compile

In [17]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 15. Revision 6

In [19]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
Functionality Attempt 008 passes the directed SPI mode-0 sampling test, but has one timing issue.

During the 32-bit payload transfer:
- All 32 MISO bits are correctly inverted before rising SCK edges.
- After rising-edge updates settle, only 31 bits remain inverted.
- The final bit returns to normal immediately after the
  32nd rising edge because the counter reaches 32.

Correct this so that:

1. All 32 payload bits remain inverted through their
   respective rising SCK edges.
2. The final payload bit remains inverted until the
   transaction ends at CS deassertion.
3. The trigger transaction remains completely unaffected.
4. Only the next transaction is corrupted.
5. Subsequent transactions return to normal.
6. Preserve all existing passing behavior and connections.

Make the smallest necessary state-machine correction.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/functionality/attempt_009.sv


## 16. Compile

In [20]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS
